In [ ]:
import scanpy as sc
import seaborn as sns
from glob import glob
from tqdm import tqdm
from multiprocessing import Pool

# 批量输出单个指标值

In [ ]:
files = glob('/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/*')

In [ ]:
def process_files(file):
    sample_name = file.split('/')[-1].split('.')[0]
    tmp = sc.read_h5ad(file)
    tmp.var["hb"] = tmp.var_names.str.contains(("^HB[^(P)]"))
    sc.pp.calculate_qc_metrics(tmp, qc_vars=["hb"], inplace=True)
    #fig = sns.displot(tmp.obs.query("total_counts_hb > 0 and total_counts_hb < 50").total_counts_hb)
    #fig.savefig(f'/home/liyanguo/MyImmuCell/02_Read_QC/hb_QC/{sample_name}.png', dpi = 400)
    fig = sns.displot(tmp.obs.query("percent_hemo > 0 and percent_hemo < 100").percent_hemo)
    fig.savefig(f'/home/liyanguo/MyImmuCell/02_Read_QC/hb_QC/{sample_name}_percent_hemo.png', dpi = 400)

In [ ]:
with Pool(72) as p:
    p.map(process_files, files)

In [ ]:
files = glob('/localdisk/immune/burn_sc_analysis/02_Read_QC/RNA_h5ad/*')

In [ ]:
def process_files(file):
    sample_name = file.split('/')[-1].split('.')[0]
    tmp = sc.read_h5ad(file)
    tmp.var["hb"] = tmp.var_names.str.contains(("^HB[^(P)]"))
    sc.pp.calculate_qc_metrics(tmp, qc_vars=["hb"], inplace=True)
    #fig = sns.displot(tmp.obs.query("total_counts_hb > 0 and total_counts_hb < 50").total_counts_hb)
    #fig.savefig(f'/localdisk/immune/burn_sc_analysis/02_Read_QC/hb_QC/{sample_name}.png', dpi = 400)
    fig = sns.displot(tmp.obs.query("percent_hemo > 0 and percent_hemo < 100").percent_hemo)
    fig.savefig(f'/localdisk/immune/burn_sc_analysis/02_Read_QC/hb_QC/{sample_name}_percent_hemo.png', dpi = 400)

In [ ]:
with Pool(48) as p:
    p.map(process_files, files)

# 单个配对样本的快速分析

In [ ]:
marker_dict1={
    'Immune cell': ['PTPRC'],

    'Lineage':['CD7','CD3E',
               'IL7R',
               'SPON2','KLRF1',
               'CD79A','MS4A1',
               'CD14','FCGR3A',
               'CSF3R','FCGR3B'
              ],
    
    'HSPC':['CD34','SPINK2','CYTL1','PROM1','SMIM24','EGFL7',
            'SOX4','KIT', 'DNTT','ETV6','MCL1','STAT5A','CD48'],
    'Basophil':['HDC','GATA2', 'FCER1A', 'IL3RA','ENPP3','MS4A2','IL4'],
    'Eosinophil':['ALOX15','SIGLEC10','SIGLEC8','LMO4','EPX','ITGA1','CCR3',],
    'Mast cell':['FCER1A','MS4A2','KIT'],

    'CEACAM8- Neutrophil':['FCGR3B','CSF3R','MME','G0S2','MNDA',],
    'CEACAM8+ Neutrophil':['CEACAM8','LTF','BPI','MPO','ELANE'],
    
    'Classical monocytes' :['CD14','LYZ','VCAN','FCN1',],
    'Non-classical monocytes' :['FCGR3A','CDKN1C','TCF7L2','CSF1R',],
    'DC':['ENHO','CD1C','HLA-DQA1','FCER1A','CLEC10A','CLEC9A','FLT3'],
    'pDC':['CLEC4C','IL3RA'],
    
    'T naive': ['CD3D','CD3G','LEF1','IL7R','CCR7', 'TCF7','SELL','BACH2'],
    'CD4+ T': ['CD4','MAL','RCAN3','IL6ST','TRAT1','CAMK4'],
    'TRAV1-2- CD8+ T': ['CD8A', 'CD8B','LINC02446','CCL5','GZMH','GZMK','ZNF683','THEMIS'],
    'γδ T':['TRDV2','TRGV9','TRDC','KLRG1','TRGC1','TRGC2','CCL5','CST7',],
    'MAIT':['SLC4A10','KLRB1','TRAV1-2','RORA','CXCR6'],
    'Treg':['FOXP3','CTLA4','IL2RA','TIGIT','RTKN2','STAM'],

    'NK': ['KLRD1','GNLY','PRF1','GZMB','CD244','CD247',
           'IL2RB','XCL1','XCL2'],
    'CD56dim':['FCGR3A','SPON2',],#CD16+
    'CD56bright':['IL7R','GZMK','NCAM1','GATA3'],#CD16-/low
    'Adaptive NK':['KLRC2','B3GAT1',],

    'Lymphocyte relate':['RUNX1','RUNX2','KLRF1','KLRC1',#KLRC1=NKG2AA抑制   NKp80=KLRF1
                         'KLRD1','KLRG1','EOMES',
                        'NCR2','NCR3','SYNE1'],#KLRD1=CD94抑制 KLRG1抑制

    'ILC': ['IL7R','LTB','RGS1','TNFSF10','PLCG2','RUNX1','TOX','FLT3'],#lin-
    'pILC':['NFIL3',],
    'ILC1':['IL2RB','TBX21','NCR1',],#CD56-
    'ILC2':['GATA3','IL2RA','PTGDR2','KLRG1','ZBTB16','RORA','MAF'],#lin- CD4-
    'ILC3-NCR+':['AHR','KIT','RORC','NCR1','NCR2','RUNX2',],#mature ILC #lin- CD56+ CD4-
    'LTi':['CCR7','KIT','ITGA4',],#mature ILC #lin- CD56-
    'ILCreg':['SOX4',],#KLRG1- 
    
    'NKT':['TRAV24','TRBV28','CLDND1'], 
    'DN T':['FXYD2','NUCB2','MYB',],
    'T Develop':['SPI1','ZBTB7B','RUNX3'],
    'T activation': ['CD69', 'CD38'],

    'B naive': ['CD79A','TCL1A',],
    'Transitional B': ['EBF1','BACH2','PAX5', 'MSI2',],
    'Atypical memory B': ['TBX21','ITGAX','FCRL5','SIGLEC6','SOX5'],
    'Plasma':['JCHAIN','IGHA1','TNFRSF17','MZB1','CD38','XBP1', 'PRDM1'],
    'CD5+ B':['IGHV3-7','LEF1','CTLA4','CD5'],

    'Platelet':['PF4','PPBP','GP9','CAVIN2'],
    'RBC':['HBB','HBA2','BLVRB'],
    'Proliferative signal':['MKI67','TOP2A','STMN1'],
    'Other':['ITGAM','ANK3','PRKCA','BCL6','TGFB1','CXCR4','GATA1','ATXN1','SELPLG',
             'ITGA6','CSF1','CSF1R','CSF2','ETS1','CD44','IRF4','STAT3','BATF','TGFBR1','TGFBR2','CD24']
}
#Protein :       CD114,  CD116,  CD123, CD124, CD125,  CD126, CD203c,  KLRB1,  CD25,   CD29,   CD73,  CD127,  CD119,  CRTH2,  PLZF, CD62L, TdT, CD57,     ThpOK,   PU.1, CRTH2
#Gene Symbol :	CSF3R,   CSF2RA, IL3RA, IL4R,  IL5RA,  IL6R, ENPP3,   CD161,  IL2RA,  ITGB1,  NT5E,  IL7R,  IFNGR1,  PTGDR2, ZBTB16, SELL, DNTT, B3GAT1, ZBTB7B,   SPI1, PTGDR2

#Protein :    CD42a, CD42b, 
#Gene Symbol :GP9, GP1BA

#TCR : Vα7.2
#Gene : TRAV1-2

#GCSF=CSF3, for neutrophils
#GM-CSF=CSF2, granulocyte-macrophage colony-stimulating factor
#M-CSF=CSF1, macrophage colony-stimulating factor

In [ ]:
sample = 'D0689_Rep2'

In [ ]:
high = sc.read_h5ad(f"/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/{sample}_high_nCount_RNA.h5ad")
low = sc.read_h5ad(f"/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/{sample}_low_nCount_RNA.h5ad")

sc.pp.normalize_total(high, target_sum=1e4)
sc.pp.log1p(high)

sc.pp.normalize_total(low, target_sum=1e4)
sc.pp.log1p(low)

sc.pp.highly_variable_genes(high,flavor='seurat_v3',n_top_genes=2000)

sc.pp.highly_variable_genes(low,flavor='seurat_v3',n_top_genes=2000)

sc.pp.pca(high)
sc.pp.pca(low)

sc.pp.neighbors(high)
sc.pp.neighbors(low)

sc.tl.leiden(high)
sc.tl.leiden(low)

In [ ]:
high.obs['Reference_Atlas_L1L2'].value_counts()

In [ ]:
high

In [ ]:
low.obs['Reference_Atlas_L1L2'].value_counts()

In [ ]:
low

In [ ]:
sc.tl.umap(high)
sc.tl.umap(low)

In [ ]:
sc.pl.umap(high,color='leiden')

In [ ]:
sc.pl.umap(low,color='leiden')

In [ ]:
sc.pl.dotplot(
        low,
        groupby='leiden',
        var_names=marker_dict1,
        standard_scale="var",
        dot_min=0.1,
        show=False
    )

In [ ]:
low.obs['leiden'].value_counts()

In [ ]:
sc.pl.dotplot(
        high,
        groupby='leiden',
        var_names=marker_dict1,
        standard_scale="var",
        dot_min=0.1,
        show=False
    )

In [ ]:
high.obs['leiden'].value_counts()

In [ ]:
high.obs['leiden'].value_counts()

In [ ]:
high

## 单个样本质控指标分析

In [ ]:
nCount_RNA_low=505
nCount_RNA_high=2868
nFeature_RNA_low=347
nFeature_RNA_high=1302

In [ ]:
tmp = low.copy()

In [ ]:
tmp.var["hb"] = tmp.var_names.str.contains(("^HB[^(P)]"))

In [ ]:
sc.pp.calculate_qc_metrics(
    tmp, qc_vars=["hb"], inplace=True, percent_top=[20], log1p=True
)

In [ ]:
data = (tmp.obs["nFeature_RNA"] > nFeature_RNA_low) & (
    tmp.obs["nFeature_RNA"] < nFeature_RNA_high) & (
    tmp.obs["percent_mito"] < 10) & (tmp.obs["nCount_RNA"] > nCount_RNA_low) & (
    tmp.obs["nCount_RNA"] < nCount_RNA_high)

In [ ]:
data.value_counts()

In [ ]:
sns.displot(
    tmp.obs.query("total_counts_hb > 0 and total_counts_hb < 100").total_counts_hb
)